
Tiếp theo hãy giúp tôi xây dựng thêm InventorySystem sao cho:
- Có thể nhặt được nhiều loại đồ khác nhau (Items, Consumables, Materials) và phân biệt chúng trong kho đồ.
- Có thể cập nhật và thay đổi cách chỉ số của vật phẩm khi bị
- Có thể xoá một món đồ cụ thể trong kho đồ dựa trên ID hoặc vị trí (slot) của nó (đặt giả nâng cấp bị thất bại và vũ khí bị phá huỷ)
- Tích hợp các hàm để sau này có thể dễ dàng liên kết với hệ thống Crafting, GemCrafting, Enchanting, .... để nâng cấp đồ hoặc thay đổi chỉ số đồ.
- Tích hợp các hàm để sau này có thể dễ dàng liên kết với hệ thống giao dịch giữa người chơi (Player Trading) hoặc chợ đen (Black Market) để mua bán đồ với nhau.
- Thêm các lệnh để test các chức năng trên trong chat (VD: /addconsumable, /addmaterial, /removeitem, /upgradeitem, /tradeitem, ...)

ItemData:
-- File: Shared/Components/ItemData.luau
local ItemDatabase = require(game.ReplicatedStorage.Shared.Constants.ItemData)
	local HttpService = game:GetService("HttpService")
-- Định nghĩa kiểu dữ liệu 
export type Enchant = {
	Id: string, 
	Level: number
}

-- Mẫu dữ liệu cho 1 lần nâng cấp
export type UpgradeEntry = {
	Material: string,            -- Loại tinh thể/vật liệu dùng
	StatChanges: {[string]: number}, -- Các chỉ số được cộng (VD: {Damage = 2, Crit = 5})
	Time: number            -- Lưu thời điểm để sau này làm log
}


export type ItemData = {
	UUID: string,             -- ID độc nhất của cục đồ này (VD: "item_982hjsdf-2342")
	ID: string,       -- Tên của đồ gốc (VD: "IronSword" -> để tra trong Constants/Weapons)

	-- Các chỉ số bị biến đổi (Chỉ tồn tại khi có sự thay đổi)
	CustomName: string?,      -- Dấu "?" nghĩa là có thể bị nil (không lưu vào DataStore nếu trống)
	CustomLore: string?,

	UpgradeCount: number?,    -- Thay cho UpgradeAttemptsUsed, nếu = 0 thì gán bằng nil luôn
	Purity: number?,
	Corruption: number?,

	OwnerId: number?,         -- UserId của chủ sở hữu (nếu là đồ bị khóa)
	BindState: "Unlocked" | "BoundToCharacter" | "BoundToAccount", -- Dễ đọc, dễ hiểu

	-- Dùng Array để lưu, không giới hạn số slot cứng.
	-- Server sẽ check `#Enchants < MaxEnchantSlots` lấy từ Template gốc.
	Enchants: {Enchant}?,     
	Gems: {string}?,          -- Chỉ cần lưu mảng ID của các viên ngọc (VD: {"Ruby", "Sapphire"})
	UpgradeHistory: {UpgradeEntry}?,
	
	State: {string}?,
	NBT: {[string]: any}?,     -- Lưu data tùy ý (Custom Data) cho các event đặc biệt
	Aura: number?
}

-- Trả về 1 hàm Factory để tạo vật phẩm mới cực kỳ sạch sẽ
local ItemFactory = {}

-- [MỚI] Thêm dấu "?" vào `ItemData?` vì hàm này có thể trả về nil nếu ID nhập vào bị sai
function ItemFactory.createNewItem(Id: string): ItemData?
    
    -- 1. KIỂM TRA TÍNH HỢP LỆ CỦA ID TỪ DATABASE
    if not ItemDatabase.Registry[Id] then
        warn("🚨 [ItemFactory] Từ chối tạo đồ! ID vật phẩm không tồn tại trong GlobalRegistry: " .. tostring(Id))
        return nil
    end

    -- 2. NẾU HỢP LỆ, TIẾN HÀNH TẠO NHỮNG DATA BẮT BUỘC
    local newItem: ItemData = {
        UUID = HttpService:GenerateGUID(false),
        ID = Id,
        BindState = "Unlocked",
    }

    return newItem
end

return ItemFactory

StackableData:
-- File: Shared/Components/StackableData.luau

export type StackableData = {
	Id: string,     -- ID gốc (VD: "HealthPotion", "IronOre")
	Amount: number,         -- Số lượng
	NBT: {[string]: any}?   -- Dành cho trường hợp đặc biệt (ví dụ: bình máu có hạn sử dụng)
}

local StackableFactory = {}

function StackableFactory.create(Id: string, amount: number): StackableData
	return {
		Id = Id,
		Amount = amount or 1,
	}
end

return StackableFactory

UpgradeTemplate:
-- File: Shared/Components/UpgradeTemplate.luau

-- Điều kiện ảnh hưởng tỉ lệ thành công
export type UpgradeCondition = {
	StatName: string,       -- Ví dụ: "Purity", "Corruption"
	RatePerPoint: number,   -- Lệch % mỗi điểm (VD: -0.1 = -10%)
	MaxCap: number,         -- Giới hạn tối đa (VD: -0.9)
}

-- [POOL PHỤ] Roll thông số
export type StatRollEntry = {
	StatName: string,
	Min: number,
	Max: number,
	LevelScale: {Min: number, Max: number}?, -- Dành cho việc scale theo level (thay cho "7:75")
	Weight: number, -- Trọng số trong pool phụ
}

-- [POOL CHÍNH] Các kịch bản có thể xảy ra khi nâng cấp
export type PoolOutcome = {
	Weight: number,                  -- Trọng số của kịch bản này trong pool chính

	-- Danh sách các pool phụ (Sẽ quay random bên trong này tiếp)
	StatPools: {StatRollEntry}?,     

	-- Các hiệu ứng và trạng thái áp dụng thẳng
	ApplyStates: {number}?,          -- Thay cho Value={1,2,3...}
	SpecialEffects: {string}?,       -- VD: {"ResetUpgrades", "ResetPurity"}

	Message: string,
	MessageColor: Color3,
}

-- Khối logic nâng cấp chính
export type UpgradeLogic = {
	AcceptTypes: {string},           -- VD: {"Weapon", "Armor", "All"}
	UpgradeCountCost: number,        -- Tốn bao nhiêu lượt nâng (thường là 1)

	BaseSuccessRate: number,         -- Tỉ lệ thành công gốc (0.0 -> 1.0)
	Conditions: {UpgradeCondition}?, -- Các điều kiện thay đổi tỉ lệ

	-- Các Pool Trọng Số
	SuccessPool: {PoolOutcome},      -- Nếu ép thành công thì roll bảng này
	FailPool: {PoolOutcome}?,        -- Nếu ép xịt thì roll bảng này
}

return {}